In [1]:
import numpy as np
from sympy import *

## Função gradiente e hessiano

In [2]:
# Definindo as funções
def gradient_op(variables, f):
    return Matrix([Lambda(variables, f.diff(v)) for v in variables])

def hesiano_op(variables, f):
    return Matrix([[Lambda(variables, f.diff(v,u)) for u in variables] for v in variables])

In [3]:
# Funções para o cálculo
def gradient_res(grad_f, vetor):
    results = []
    for dx in grad_f:
        results.append(dx(*vetor))
    return results

def hesiano_res(hes_f, vetor):
    matrix = []
    lin = []
    arr = np.array(hes_f)
    for i in arr:
        for j in i:
            lin.append(j(*vetor))
        matrix.append(lin)
        lin = []
    return Matrix(matrix)

In [4]:
# Função para o cálculo de módulos
def modulo(x): 
    return float(sqrt(sum(i**2 for i in x)))

### Exemplos

In [5]:
# Defini variaveis
init_printing()
variables = var('x1:3')

In [6]:
# Defini a função
f = x1**4+2*x2*x1**2+x2**2-4*x1
fl = Lambda(variables, f)
fl

In [7]:
grad = gradient_op(variables, f)
grad

⎡               3              ⎤
⎢(x₁, x₂) ↦ 4⋅x₁  + 4⋅x₁⋅x₂ - 4⎥
⎢                              ⎥
⎢                  2           ⎥
⎣   (x₁, x₂) ↦ 2⋅x₁  + 2⋅x₂    ⎦

In [8]:
hes = hesiano_op(variables, f)
hes

⎡             ⎛    2     ⎞                 ⎤
⎢(x₁, x₂) ↦ 4⋅⎝3⋅x₁  + x₂⎠  (x₁, x₂) ↦ 4⋅x₁⎥
⎢                                          ⎥
⎣     (x₁, x₂) ↦ 4⋅x₁        (x₁, x₂) ↦ 2  ⎦

## Newton Globalizado

In [9]:
# Cálculo da direção
def direcao_NG(variables, beta, grad_l, hes_l):
    mi = 0
    A = []
    while True:
        A = hes_l + mi*np.identity(len(variables))
        try:
            d = np.linalg.solve(A, -grad_l)
        
            if sum(d*grad_l) > -the*modulo(grad_l)*modulo(d):
                mi = max(2*mi, beta)
            else:
                break
        except:
            mi = max(2*mi, beta)
    return d

### Exemplo com NG

In [10]:
# Defini x0
x0 = [1,0]

In [35]:
# Defini os parametros
sig = 0.001
alp = 0.0001
gama = 0.5
the = 0.000001
beta = 0.001
M = 10000
epi = 0.1

In [36]:
# Realiza as iterções
k = 0
x = x0
while modulo(gradient_res(grad, x)) > epi and k < M:
    grad_l = np.array(gradient_res(grad, x), dtype=np.float)
    hes_l = np.array(hesiano_res(hes, x), dtype=np.float)
    
    d = direcao_NG(variables, beta, grad_l, hes_l)
            
    #fazendo as verificações de armijo
    if modulo(d) < sig*modulo(grad_l):
        d = the*(modulo(grad_l)/modulo(d))*d
        
    #calculando xk
    t = 1
    while fl(*(x+d*t)) > fl(*x) + alp*t*sum(d*grad_l):
        t = gama*t
    
    x = x+d*t
    k = k + 1

In [37]:
x, fl(*x), k

(array([    964.30419384, -929881.88302472]), -3856.73315429688, 10000)